In [ ]:
from ultralytics import YOLO
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
import cv2
import random
import yaml
import matplotlib.image as mpimg
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')


In [4]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

CUDA available: False
CUDA device count: 0


In [6]:
BASE_PATH = "../BoneFractureYolo8"

train_images = os.path.join(BASE_PATH, "train", "images")
train_labels = os.path.join(BASE_PATH, "train", "labels")

val_images = os.path.join(BASE_PATH, "valid", "images")
val_labels = os.path.join(BASE_PATH, "valid", "labels")

test_images = os.path.join(BASE_PATH, "test", "images")
test_labels = os.path.join(BASE_PATH, "test", "labels")

In [9]:
DATA_YAML = os.path.join(BASE_PATH, "data.yaml")
with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

print(data)

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus', 'shoulder fracture', 'wrist positive'], 'roboflow': {'workspace': 'veda', 'project': 'bone-fracture-detection-daoon', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/veda/bone-fracture-detection-daoon/dataset/4'}}


In [ ]:
model = YOLO("yolov8m.pt")

train_results = model.train(
    data=DATA_YAML,
    epochs=80,          
    imgsz=640,
    batch=8,           
    device="cpu" if not torch.cuda.is_available() else 0,
    workers=0,          
    patience=20,
    pretrained=True,
    optimizer="AdamW",  
    lr0=1e-3,
    cos_lr=True,
    weight_decay=5e-4,
    cache=True,
    seed=42,
    deterministic=True,
    verbose=True,
    project="runs",
    name="bone_fracture_yolov8m_merged"
)

Ultralytics 8.4.21  Python-3.12.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../BoneFractureYolo8\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

In [ ]:
best_model_path = os.path.join(
    "runs",
    "bone_fracture_yolov8m_merged",
    "weights",
    "best.pt"
)

best_model = YOLO(best_model_path)

metrics = best_model.val(
    data=DATA_YAML,
    split="test",
    device="cpu" if not torch.cuda.is_available() else 0
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Training results saved in: D:\yolo_results\train


In [ ]:
pred_results = best_model.predict(
    source=test_images,
    conf=0.25,
    save=True,
    device="cpu" if not torch.cuda.is_available() else 0,
    project="runs",
    name="bone_fracture_test_preds"
)


image 1/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\distal-humerus-fracture-1_jpg.rf.831cb137cfcbde1079f86abd5f5f2867.jpg: 640x256 1 elbow positive, 35.3ms
image 2/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\image1_0_png.rf.99862308d714bff3f9c410adf5ca93ac.jpg: 480x640 1 fingers positive, 92.8ms
image 3/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\image1_1000_png.rf.a53c5e186c03961bf88075c6e3e94cf6.jpg: 544x640 (no detections), 40.6ms
image 4/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\image1_1015_png.rf.3b7320c3c40771fa5532bf713a728b83.jpg: 544x640 (no detections), 39.1ms
image 5/169 C:\Users\Admin\Desktop\graduate study\Deep learning\..\BoneFractureYolo8\test\images\image1_1015_png.rf.9181f8eb07451331e22381bacb3a5bd2.jpg: 640x640 (no detections), 32.5ms
image 6/169 C:\Users\Admin\Desktop\graduate study\Deep

In [ ]:
run_dir = Path("runs") / "bone_fracture_yolov8m_merged"

# confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cm_path = run_dir / "confusion_matrix.png"
cmn_path = run_dir / "confusion_matrix_normalized.png"

if cm_path.exists():
    axes[0].imshow(mpimg.imread(cm_path))
    axes[0].set_title("Confusion Matrix")
    axes[0].axis("off")
else:
    axes[0].text(0.5, 0.5, "confusion_matrix.png not found", ha="center", va="center")
    axes[0].axis("off")

if cmn_path.exists():
    axes[1].imshow(mpimg.imread(cmn_path))
    axes[1].set_title("Normalized Confusion Matrix")
    axes[1].axis("off")
else:
    axes[1].text(0.5, 0.5, "confusion_matrix_normalized.png not found", ha="center", va="center")
    axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# results.csv
results_csv = run_dir / "results.csv"
results_df = pd.read_csv(results_csv)
results_df.columns = [c.strip() for c in results_df.columns]



In [ ]:
# plot losses
loss_cols = [c for c in results_df.columns if "loss" in c.lower()]
plt.figure(figsize=(10, 6))
for col in loss_cols:
    plt.plot(results_df["epoch"], results_df[col], label=col)
plt.title("Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



In [ ]:
# plot main metrics
metric_cols = [
    c for c in [
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)"
    ] if c in results_df.columns
]

plt.figure(figsize=(10, 6))
for col in metric_cols:
    plt.plot(results_df["epoch"], results_df[col], label=col)
plt.title("Validation Metrics per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.legend()
plt.grid(alpha=0.3)
plt.show()